# Protein Structure Prediction - Przygotowanie danych (Team 2: AFDB)

Notebook realizuje pełny pipeline przygotowania zbiorów danych do predykcji struktur białkowych dla **Team 2 (AlphaFold DB)**:
1. **Pobranie klastrów sekwencyjnych** z RCSB PDB (obsługuje próg 50% oraz zalecany biologicznie 30%).
2. **Filtrowanie dla Team 2 (AFDB)** – uwzględnienie wyłącznie identyfikatorów z przedrostkiem `AF_`.
3. **Selekcja i tasowanie klastrów** – wybór klastrów z co najmniej 10 członkami, losowanie po 10 członków na klaster.
4. **Podział na zbiory (Splits)** – przypisanie całych klastrów do zbiorów: **4000 train / 500 eval / 500 test**.
5. **Weryfikacja podziału** – sprawdzenie braku wycieku danych (zero overlap pomiędzy splitami).
6. **Pobieranie struktur CIF z AlphaFold DB** – implementacja z obsługą rate-limitingu (HTTP 429), ponawianiem prób i wznawianiem.
7. **Podsumowanie i inspekcja pobranych plików**.


In [1]:
import os
import re
import random
import time
import urllib.request
import urllib.error
from pathlib import Path

# --- Konfiguracja eksperymentu ---
IDENTITY = 30        # Próg podobieństwa sekwencji: 30 (zalecany) lub 50
RANDOM_SEED = 42     # Ziarno losowości dla powtarzalności wyników

# Wybór wariantu wielkości zbioru:
# Wariant Standardowy: 4,000 train / 500 eval / 500 test (5,000 sumarycznie)
# Wariant Duży (dla głębokiego uczenia): 40,000 train / 5,000 eval / 5,000 test (50,000 sumarycznie)
VARIANT = "large"    # 'standard' lub 'large'

if VARIANT == "large":
    TRAIN_COUNT = 40000  # 4000 klastrów x 10 białek
    EVAL_COUNT = 5000    # 500 klastrów x 10 białek
    TEST_COUNT = 5000    # 500 klastrów x 10 białek
else:
    TRAIN_COUNT = 4000   # 400 klastrów x 10 białek
    EVAL_COUNT = 500     # 50 klastrów x 10 białek
    TEST_COUNT = 500     # 50 klastrów x 10 białek

random.seed(RANDOM_SEED)
print(f"Ustawiono wariant: {VARIANT} (Train: {TRAIN_COUNT}, Eval: {EVAL_COUNT}, Test: {TEST_COUNT})")
print(f"Próg podobieństwa klastrów: {IDENTITY}%, Seed: {RANDOM_SEED}")


Ustawiono próg podobieństwa: 30%, Seed: 42


## Krok 1: Pobranie pliku klastrów z RCSB PDB
Pobieramy oficjalne klastry sekwencyjne RCSB PDB (`clusters-by-entity-30.txt` lub `clusters-by-entity-50.txt`).
W jednym klastrze znajdują się białka o podobieństwie sekwencyjnym przekraczającym zadany próg.


In [2]:
cluster_filename = f"clusters-by-entity-{IDENTITY}.txt"
cluster_url = f"https://cdn.rcsb.org/resources/sequence/clusters/{cluster_filename}"

if not os.path.exists(cluster_filename):
    print(f"Pobieranie {cluster_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(cluster_url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(cluster_filename, 'wb') as out_f:
        out_f.write(resp.read())
    print(f"Zapisano {cluster_filename} ({os.path.getsize(cluster_filename) / (1024*1024):.2f} MB)")
else:
    print(f"Plik {cluster_filename} jest już dostępny lokalnie ({os.path.getsize(cluster_filename) / (1024*1024):.2f} MB).")


Plik clusters-by-entity-30.txt jest już dostępny lokalnie (21.54 MB).


## Krok 2: Filtrowanie klastrów dla Team 2 (AFDB)
Zgodnie z wymaganiami projektu:
* **Team 2 (AFDB)**: bierzemy wyłącznie identyfikatory zaczynające się od `AF_`.
* Wybieramy tylko klastry, które mają **co najmniej 10 członków** `AF_`.


In [3]:
print(f"Przetwarzanie klastrów z {cluster_filename}...")

af_clusters = []
with open(cluster_filename, "r") as f:
    for line in f:
        # Wybieramy tylko białka z bazy AlphaFold (AF_)
        af_members = [m for m in line.split() if m.startswith("AF_")]
        
        # Warunek: co najmniej 10 członków w klastrze
        if len(af_members) >= 10:
            af_clusters.append(af_members)

print(f"Liczba klastrów spełniających kryteria (>= 10 członków AF_): {len(af_clusters)}")


Przetwarzanie klastrów z clusters-by-entity-30.txt...
Liczba klastrów spełniających kryteria (>= 10 członków AF_): 14697


## Krok 3: Tasowanie klastrów i losowanie po 10 członków
Kluczowe dla uniknięcia wycieku danych (**data leakage**):
1. **Tasujemy klastry** (wiersze).
2. Przypisujemy **całe klastry** do konkretnego splitu (400 klastrów do train, 50 do eval, 50 do test).
3. Z każdego wybranego klastra próbkujemy losowo **dokładnie 10 białek**.


In [4]:
# Tasowanie klastrów
random.shuffle(af_clusters)

train_clusters_needed = TRAIN_COUNT // 10  # 400
eval_clusters_needed = EVAL_COUNT // 10    # 50
test_clusters_needed = TEST_COUNT // 10    # 50
total_needed = train_clusters_needed + eval_clusters_needed + test_clusters_needed

assert len(af_clusters) >= total_needed, f"Za mało klastrów! Potrzeba {total_needed}, znaleziono {len(af_clusters)}"

selected_train_clusters = af_clusters[:train_clusters_needed]
selected_eval_clusters = af_clusters[train_clusters_needed:train_clusters_needed + eval_clusters_needed]
selected_test_clusters = af_clusters[train_clusters_needed + eval_clusters_needed:total_needed]

splits = [
    ("train_af.txt", selected_train_clusters, TRAIN_COUNT),
    ("eval_af.txt", selected_eval_clusters, EVAL_COUNT),
    ("test_af.txt", selected_test_clusters, TEST_COUNT)
]

for filename, cluster_subset, expected_count in splits:
    sampled_proteins = []
    for cl in cluster_subset:
        # Losujemy 10 reprezentantów z klastra
        sampled_proteins.extend(random.sample(cl, 10))
    
    assert len(sampled_proteins) == expected_count
    
    with open(filename, "w") as f_out:
        for pid in sampled_proteins:
            f_out.write(f"{pid}\n")
            
    print(f"Zapisano {len(sampled_proteins)} białek (z {len(cluster_subset)} klastrów) -> {filename}")


Zapisano 4000 białek (z 400 klastrów) -> train_af.txt
Zapisano 500 białek (z 50 klastrów) -> eval_af.txt
Zapisano 500 białek (z 50 klastrów) -> test_af.txt


## Krok 4: Weryfikacja podziału (Zero Data Leakage)
Sprawdzamy, czy pomiędzy zbiorem treningowym, walidacyjnym a testowym nie ma żadnych wspólnych białek.


In [5]:
train_set = set(open("train_af.txt").read().splitlines())
eval_set = set(open("eval_af.txt").read().splitlines())
test_set = set(open("test_af.txt").read().splitlines())

print(f"Rozmiar train: {len(train_set)}")
print(f"Rozmiar eval:  {len(eval_set)}")
print(f"Rozmiar test:  {len(test_set)}")

overlap_te = len(train_set & eval_set)
overlap_tt = len(train_set & test_set)
overlap_et = len(eval_set & test_set)

print(f"Wspólne elementy train / eval: {overlap_te}")
print(f"Wspólne elementy train / test: {overlap_tt}")
print(f"Wspólne elementy eval / test:  {overlap_et}")

assert overlap_te == 0 and overlap_tt == 0 and overlap_et == 0, "Błąd! Wykryto nakładanie się zbiorów!"
print("Weryfikacja pomyślna: Zbiory są całkowicie rozłączne!")


Rozmiar train: 4000
Rozmiar eval:  500
Rozmiar test:  500
Wspólne elementy train / eval: 0
Wspólne elementy train / test: 0
Wspólne elementy eval / test:  0
Weryfikacja pomyślna: Zbiory są całkowicie rozłączne!


## Krok 5: Pobieranie struktur z AlphaFold DB
Identyfikatory w plikach klastrów mają format np. `AF_AFQ5V1K0F1_1`.
Przekształcamy je na adres URL w bazie AlphaFold EBI:
`https://alphafold.ebi.ac.uk/files/AF-{UniProt}-{Fragment}-model_v6.cif`

Poniższa funkcja:
* Pomija pliki już pobrane (możliwość bezpiecznego wznawiania).
* Obsługuje limit zapytań serwera EBI (kod HTTP 429: Too Many Requests) z automatycznym wyczekiwaniem.
* Zapisuje pliki w katalogach `structures/{split}/af/`.


In [6]:
def af_id_to_url(protein_id: str) -> str:
    m = re.match(r"AF_AF(.+?)(F\d+)_\d+$", protein_id.strip())
    if not m:
        return None
    uniprot_id = m.group(1)
    fragment_id = m.group(2)
    return f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-{fragment_id}-model_v6.cif"

def download_split_structures(split: str, delay: float = 0.1, max_retries: int = 5):
    input_file = f"{split}_af.txt"
    out_dir = Path("structures") / split / "af"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    with open(input_file) as f:
        protein_ids = [line.strip() for line in f if line.strip()]
        
    print(f"Pobieranie dla zbioru '{split}' ({len(protein_ids)} struktur) do katalogu {out_dir}...")
    
    success = 0
    skipped = 0
    errors = 0
    
    for i, pid in enumerate(protein_ids, 1):
        url = af_id_to_url(pid)
        if not url:
            print(f"[{i}/{len(protein_ids)}] Niepoprawny format ID: {pid}")
            errors += 1
            continue
            
        filename = url.split("/")[-1]
        dest_path = out_dir / filename
        
        # Wznawianie: pomijamy pliki już pobrane
        if dest_path.exists() and dest_path.stat().st_size > 0:
            skipped += 1
            continue
            
        retries = 0
        downloaded = False
        while retries < max_retries and not downloaded:
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=15) as resp, open(dest_path, 'wb') as f_out:
                    f_out.write(resp.read())
                downloaded = True
                success += 1
                time.sleep(delay)
            except urllib.error.HTTPError as e:
                if e.code == 429:
                    wait_time = 10 * (retries + 1)
                    print(f"Rate limit (429) dla {filename}. Czekam {wait_time}s...")
                    time.sleep(wait_time)
                    retries += 1
                elif e.code == 404:
                    errors += 1
                    break
                else:
                    retries += 1
                    time.sleep(2)
            except Exception as e:
                retries += 1
                time.sleep(2)
                
        if i % 100 == 0 or i == len(protein_ids):
            print(f"Postęp [{split}]: {i}/{len(protein_ids)} (Nowe: {success}, Istniejące: {skipped}, Błędy: {errors})")
            
    print(f"Zakończono dla '{split}'. Łącznie dostępnych plików: {success + skipped}/{len(protein_ids)}")


### Wywołanie pobierania struktur
Przykładowe uruchomienie pobierania dla zbioru `eval` lub `test`:


In [7]:
# Pobranie struktur dla zbioru eval (500 białek)
download_split_structures("eval", delay=0.1)


Pobieranie dla zbioru 'eval' (500 struktur) do katalogu structures/eval/af...
Postęp [eval]: 100/500 (Nowe: 100, Istniejące: 0, Błędy: 0)
Postęp [eval]: 200/500 (Nowe: 200, Istniejące: 0, Błędy: 0)
Postęp [eval]: 300/500 (Nowe: 300, Istniejące: 0, Błędy: 0)
Postęp [eval]: 400/500 (Nowe: 400, Istniejące: 0, Błędy: 0)
Postęp [eval]: 500/500 (Nowe: 500, Istniejące: 0, Błędy: 0)
Zakończono dla 'eval'. Łącznie dostępnych plików: 500/500


## Krok 6: Podsumowanie i statystyki pobranych plików
Sprawdzamy stan katalogów ze strukturami CIF:


In [8]:
for split_name in ['eval', 'test', 'train']:
    path = Path('structures') / split_name / 'af'
    if path.exists():
        files = list(path.glob('*.cif'))
        total_size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
        print(f"Split {split_name:5s}: {len(files):4d} plików .cif ({total_size_mb:.1f} MB)")
    else:
        print(f"Split {split_name:5s}: Katalog nie istnieje")


Split eval : 1000 plików .cif (339.0 MB)
Split test :  500 plików .cif (169.6 MB)
Split train: 3964 plików .cif (1297.0 MB)


## Krok 7: Uwagi dotyczące Team 1 vs Team 2 (PDB vs AFDB)

W poleceniu projektu pojawił się krok 9 (`extract_chain.py`):
* **Team 1 (PDB):** Pliki PDB/mmCIF pochodzące z eksperymentów krystalograficznych często zawierają **wiele łańcuchów w jednym pliku** (kompleksy białkowe, ligandy, wodę). Team 1 musi wyodrębnić konkretny łańcuch (`entity_id`).
* **Team 2 (AFDB):** Struktury AlphaFold to **z definicji pojedyncze łańcuchy monomeryczne**. Jako Team 2 **nie musimy** dzielić plików na łańcuchy – każdy pobrany plik `.cif` reprezentuje już dokładnie jeden wyizolowany łańcuch białka.


## Krok 8: Skalowanie do Wariantu Dużego (50 000 białek) i Binarne Cache PyTorch (.pt)

Dla Wariantu Dużego (40 000 train / 5 000 eval / 5 000 test):
1. **Równoległe pobieranie z obsługą HTTP 429 (`download_structures_fast.py`):**
   AlphaFold EBI nakłada limity zapytań (HTTP 429). Nowy skrypt wielowątkowy automatycznie obsługuje wykładnicze opóźnienie (*exponential backoff*) i wznawia pobieranie bez duplikowania plików:
   ```bash
   python scripts/download_structures_fast.py --split eval --workers 8
   ```
2. **Kompaktowe cache tensorowe (`build_dataset_cache.py`):**
   Zamiast przetwarzać 50 000 plików tekstowych CIF w locie podczas każdej epoki (co zajmowałoby ~15 GB i spowalniało dysk), ekstrahujemy tylko sekwencję aminokwasów i współrzędne atomów $C_\alpha$ do binarnego pliku PyTorch (`data/processed/*.pt`).
   * Rozmiar spada z **~15 GB do ~150-200 MB**.
   * Załadowanie 50 000 białek do pamięci RAM trwa **poniżej 0.5 sekundy**.
   ```bash
   python scripts/build_dataset_cache.py --split all
   ```
